# Clase 197 — CI/CD para ML con GitHub Actions

Notebook **declarativo**: muestra los YAML de workflow y scripts del repo objetivo. Para verlos en vivo, copialos a un repo GitHub y abrí un PR.

Estructura del repo:
```
.github/workflows/ml.yml
src/train.py · src/evaluate.py · src/diff_metrics.py · src/check_threshold.py
tests/test_train.py
params.yaml · requirements.txt
```

## 1. Workflow `ml.yml`

In [ ]:
workflow = '''\
name: ml
on:
  pull_request: { branches: [main] }
  push: { branches: [main] }
permissions:
  contents: read
  pull-requests: write
  id-token: write   # OIDC para AWS
jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - run: pip install ruff && ruff check src tests
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - run: pip install -r requirements.txt && pytest -q
  train-and-report:
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    needs: [lint, test]
    steps:
      - uses: actions/checkout@v4
        with: { fetch-depth: 0 }
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - uses: iterative/setup-cml@v2
      - run: pip install -r requirements.txt
      - run: python src/train.py && python src/evaluate.py > metrics_pr.json
      - name: Train on main for comparison
        run: |
          git checkout origin/main -- src/
          python src/train.py && python src/evaluate.py > metrics_main.json
          git checkout HEAD -- src/
      - name: Comment report
        env: { REPO_TOKEN: "${{ secrets.GITHUB_TOKEN }}" }
        run: |
          python src/diff_metrics.py metrics_main.json metrics_pr.json > report.md
          cml comment create report.md
      - run: python src/check_threshold.py metrics_main.json metrics_pr.json 0.03
'''
print(workflow)

## 2. `diff_metrics.py` y `check_threshold.py`

In [ ]:
diff_metrics = '''\
import json, sys
main, pr = json.load(open(sys.argv[1])), json.load(open(sys.argv[2]))
print("## 📊 Modelo PR vs main\\n")
print("| métrica | main | PR | Δ |\\n|---|---|---|---|")
for k in sorted(set(main) | set(pr)):
    a, b = main.get(k, 0), pr.get(k, 0)
    print(f"| {k} | {a:.4f} | {b:.4f} | {'🟢' if b >= a else '🔴'} {b - a:+.4f} |")
'''
check = '''\
import json, sys
main, pr, tol = json.load(open(sys.argv[1])), json.load(open(sys.argv[2])), float(sys.argv[3])
regr = [f"{k}: {main[k]:.4f} -> {pr[k]:.4f}" for k in main if k in pr and pr[k] < main[k] - tol]
if regr: print("REGRESSION:", *regr, sep="\\n"); sys.exit(1)
print("OK")
'''
print(diff_metrics)
print('---')
print(check)

## 3. Simulación local: 2 modelos comparados

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)

def eval_model(m, name):
    p = m.fit(Xtr, ytr).predict(Xte)
    return {'model': name, 'accuracy': accuracy_score(yte, p), 'f1_macro': f1_score(yte, p, average='macro')}

main_m = eval_model(RandomForestClassifier(n_estimators=100, random_state=42), 'rf-100')
pr_m = eval_model(LogisticRegression(max_iter=500), 'lr')
print('main:', main_m); print('pr:  ', pr_m)

print('\n## Δ vs main')
for k in ['accuracy', 'f1_macro']:
    d = pr_m[k] - main_m[k]
    print(f'  {k}: {d:+.4f}  {"🟢" if d >= 0 else "🔴"}')

## Ejercicio guiado

1. Copiá el workflow a un repo real. Hacé un PR con `RandomForest(n_estimators=5)` (peor). Confirmá comment CML + check rojo.
2. Activá branch protection en `main` con `lint`, `test`, `train-and-report` como required.
3. Bonus: job `deploy` solo en `push: main`, con OIDC a AWS y `aws s3 cp model.pkl s3://bucket/`.

## Conclusiones

- CI/CD convierte "el modelo nuevo es mejor" en un check ejecutable.
- Cache de pip + datasets corta CI de 25 min a 3 min.
- OIDC reemplaza secrets long-lived.
- Sin branch protection + required checks, el CI es sugerencia, no gate.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. CI/CD corre en la **infraestructura de GitHub Actions**, así que el YAML de workflow no es ejecutable localmente — pero sí lo **validamos con `pyyaml`** (el mismo parser que usa Actions) para garantizar que está bien formado, y ejecutamos el *concepto* de los pasos de ML (generar el reporte CML, comparar métricas vs `main`) con Python puro. Un YAML de Actions que parsea limpio + jobs con `needs`/`if` correctos es el 90% del trabajo.

In [ ]:
import yaml, json
def valid_yaml(text):
    obj = yaml.safe_load(text)        # el mismo parser de GitHub Actions
    assert isinstance(obj, dict) and 'jobs' in obj
    return obj
print('validador de workflows listo (pyyaml).')

### Ejercicio 1 — Workflow mínimo: `lint` + `test` en paralelo

Dos jobs sin `needs` entre ellos → corren en paralelo. Validamos el YAML.

In [ ]:
ci_yml = '''
name: CI
on: [push, pull_request]
jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install ruff
      - run: ruff check .
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install -r requirements.txt pytest
      - run: pytest -q
'''
obj = valid_yaml(ci_yml)
assert set(obj['jobs']) == {'lint', 'test'}
assert 'needs' not in obj['jobs']['lint'] and 'needs' not in obj['jobs']['test']
print('OK — ci.yml válido; lint y test SIN needs => corren en paralelo.')

### Ejercicio 2 — Job de `train` que sube artefactos (solo en PRs)

`if: github.event_name == 'pull_request'` restringe el job a PRs; `actions/upload-artifact` publica `model.pkl` y `metrics.json`.

In [ ]:
train_job = '''
jobs:
  train:
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install -r requirements.txt
      - run: python src/train.py
      - uses: actions/upload-artifact@v4
        with:
          name: model-and-metrics
          path: |
            model.pkl
            metrics.json
'''
job = valid_yaml(train_job)['jobs']['train']
assert job['if'] == "github.event_name == 'pull_request'"
assert any('upload-artifact' in str(s.get('uses', '')) for s in job['steps'])
print('OK — train se dispara solo en PRs y sube model.pkl + metrics.json.')

### Ejercicio 3 — Reporte CML

`iterative/setup-cml@v2` + `cml comment create report.md` postea un comentario en el PR. El YAML es declarativo, pero la **construcción del `report.md`** (tabla desde `metrics.json`) es Python puro y ejecutable.

In [ ]:
cml_step = '''
      - uses: iterative/setup-cml@v2
      - env: {REPO_TOKEN: "${{ secrets.GITHUB_TOKEN }}"}
        run: |
          echo "## Model metrics" > report.md
          cat metrics.json >> report.md
          cml comment create report.md
'''
# concepto ejecutable: generamos el report.md que CML postearía
metrics = {'accuracy': 0.912, 'f1': 0.887}
def metrics_to_md(m, title='Model metrics'):
    lines = [f'## {title}', '', '| metric | value |', '|---|---|']
    lines += [f'| {k} | {v} |' for k, v in m.items()]
    return '\n'.join(lines)
report = metrics_to_md(metrics)
print(report)
assert '| accuracy | 0.912 |' in report
print('\nOK — report.md generado (cml comment create lo postea en el PR).')

### Ejercicio 4 — Comparación de métricas contra `main`

Se entrena en la rama y en `main`, y el reporte muestra `Δaccuracy`, `Δf1`. La lógica del delta es ejecutable.

In [ ]:
metrics_pr   = {'accuracy': 0.912, 'f1': 0.887}
metrics_main = {'accuracy': 0.905, 'f1': 0.880}

def delta_report(pr, main):
    lines = ['| metric | main | PR | Δ |', '|---|---|---|---|']
    for k in pr:
        d = round(pr[k] - main[k], 4)
        arrow = '🟢' if d > 0 else '🔴' if d < 0 else '⚪'
        lines.append(f'| {k} | {main[k]} | {pr[k]} | {d:+} {arrow} |')
    return '\n'.join(lines)

rep = delta_report(metrics_pr, metrics_main)
print(rep)
assert '+0.007' in rep and '+0.007'  # Δaccuracy = 0.912 - 0.905
assert round(metrics_pr['accuracy'] - metrics_main['accuracy'], 4) == 0.007
print('\nOK — tabla de Δ contra main lista para el comentario del PR.')

### Ejercicio 5 — Branch protection (declarativo)

En **Settings → Branches → Add rule → `main`**: marcar *Require status checks to pass before merging* y seleccionar `lint`, `test`, `train`. Esto se configura en la UI/API de GitHub (no en YAML), así que la solución es la **regla** y su verificación conceptual: un PR solo mergea si TODOS los checks requeridos pasan.

In [ ]:
# API real: PUT /repos/{owner}/{repo}/branches/main/protection
branch_protection = {
    'required_status_checks': {'strict': True, 'contexts': ['lint', 'test', 'train']},
    'enforce_admins': True,
    'required_pull_request_reviews': {'required_approving_review_count': 1},
}
def can_merge(protection, checks_status: dict):
    required = protection['required_status_checks']['contexts']
    return all(checks_status.get(c) == 'success' for c in required)

assert can_merge(branch_protection, {'lint': 'success', 'test': 'success', 'train': 'success'}) is True
assert can_merge(branch_protection, {'lint': 'success', 'test': 'failure', 'train': 'success'}) is False
print('required checks:', branch_protection['required_status_checks']['contexts'])
print('OK — con test en failure el merge queda bloqueado (branch protection).')